<a href="https://colab.research.google.com/github/shayanR10/FIV1/blob/main/FIV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# README: This is my dinosaur identifier AI model, built in PyTorch via Google Colab; pretty self explanatory,
# so I'll just cut to the chase and show you how the whole thing works:
# (also make sure to run these cells in the order in which they appear in this notebook).

In [ ]:
#prerequisites, run this cell first; make sure to also create a Google Kaggle Account along with Colab secrets named "KAGGLE_USERNAME" , "KAGGLE_SLUG" , and "KAGGLE_KEY" for your Google Kaggle
# username, dataset slug, and custom API key, respectively.

!pip install ftfy regex tqdm -q
!pip install git+https://github.com/openai/CLIP.git -q
!pip install kagglehub -q

In [ ]:
#Kaggle connection setup; data pipeline

import os
from google.colab import userdata
import kagglehub

username = userdata.get('KAGGLE_USERNAME')
slug = userdata.get('KAGGLE_SLUG')
key = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

print("Authenticating")

try:
    path = kagglehub.dataset_download(f"{username}/{slug}")
    print("Success")
    print(path)
except Exception as e:
    print("Failed.")
    kagglehub.login()

In [ ]:
#image acquisition and upload pipeline

import os
import time
import requests
import torch
import clip
from urllib.parse import urlparse
from google.colab import userdata
from PIL import Image
import kagglehub
from concurrent.futures import ThreadPoolExecutor

# setup
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
kUser = userdata.get('KAGGLE_USERNAME')
kSlug = userdata.get('KAGGLE_SLUG')
dataDir = "./fossil-data"
os.makedirs(dataDir, exist_ok=True)

# Core species list
nonNegotiableDinos = [
    # Stegosauridae
    "Stegosaurus stenops", "Stegosaurus ungulatus", "Kentrosaurus aethiopicus",
    "Hesperosaurus mjosi", "Tuojiangosaurus multispinus", "Miragaia longicollum", "Dacentrurus armatus",

    # Hadrosauridae & Relatives
    "Edmontosaurus annectens", "Edmontosaurus regalis", "Saurolophus osborni",
    "Saurolophus angustirostris", "Maiasaura peeblesorum", "Brachylophosaurus canadensis",
    "Gryposaurus notabilis", "Prosaurolophus maximus", "Shantungosaurus giganteus",
    "Parasaurolophus walkeri", "Corythosaurus casuarius", "Lambeosaurus lambei",
    "Lambeosaurus magnicristatus", "Hypacrosaurus altispinus", "Hypacrosaurus stebingeri",
    "Olorotitan arharensis", "Tsintaosaurus spinorhinus", "Dryosaurus altus",
    "Ouranosaurus nigeriensis", "Iguanodon bernissartensis", "Mantellisaurus atherfieldensis",
    "Protohadros byrdi", "Probactrosaurus gobiensis", "Bactrosaurus johnsoni",
    "Eolambia caroljonesa", "Tethyshadros insularis", "Telmatosaurus transsylvanicus",

    # Sauropoda
    "Diplodocus carnegii", "Diplodocus hallorum", "Apatosaurus louisae",
    "Brontosaurus excelsus", "Dicraeosaurus hansemanni", "Amargasaurus cazaui",
    "Camarasaurus lentus", "Camarasaurus supremus", "Giraffatitan brancai",
    "Brachiosaurus altithorax", "Patagotitan mayorum", "Dreadnoughtus schrani",
    "Rapetosaurus krausei", "Saltasaurus loricatus", "Futalognkosaurus dukei",
    "Shunosaurus lii", "Mamenchisaurus hochuanensis",

    # Ankylosauridae
    "Ankylosaurus magniventris", "Zuul crurivastator", "Euoplocephalus tutus",
    "Pinacosaurus grangeri", "Saichania chulsanensis", "Tarchia kielanae",
    "Anodontosaurus lambei", "Scolosaurus cutleri", "Akainacephalus johnsoni",
    "Gobisaurus domoculus", "Shamosaurus scutatus", "Jinyunpelta sinensis",

    # Spinosauridae
    "Baryonyx walkeri", "Suchomimus tenerensis", "Spinosaurus aegyptiacus",
    "Ceratosuchops inferodios", "Riparovenator milnerae", "Irritator challengeri", "Spinomimus iguidiensis",

    # Dromaeosauridae & Troodontidae
    "Dromaeosaurus albertensis", "Utahraptor ostrommaysi", "Deinonychus antirrhopus",
    "Achillobator giganticus", "Velociraptor mongoliensis", "Tsaagan mangas",
    "Linheraptor exquisitus", "Microraptor zhaoianus", "Graciliraptor kujitang",
    "Sinornithosaurus millenii", "Changyuraptor yangi", "Buitreraptor gonzalezorum",
    "Halszkaraptor escuilliei", "Sinornithoides youngi", "Gobivenator mongoliensis", "Mei long",

    # Other Major Theropoda
    "Coelophysis bauri", "Procompsognathus triassicus", "Dilophosaurus wetherilli",
    "Ceratosaurus nasicornis", "Carnotaurus sastrei", "Majungasaurus crenatissimus",
    "Megaraptor namunhuaiquii", "Allosaurus fragilis", "Allosaurus jimmadseni",
    "Sinraptor dongi", "Monolophosaurus jiangi", "Acrocanthosaurus atokensis",
    "Giganotosaurus carolinii", "Carcharodontosaurus saharicus", "Tyrannosaurus rex",
    "Tarbosaurus bataar", "Albertosaurus sarcophagus", "Gorgosaurus libratus",
    "Daspletosaurus torosus", "Yutyrannus huali", "Guanlong wucaii",
    "Compsognathus longipes", "Sinosauropteryx prima", "Sinocalliopteryx gigas",
    "Huaxiagnathus orientalis", "Gallimimus bullatus", "Struthiomimus altus",
    "Ornithomimus velox", "Therizinosaurus cheloniformis", "Alxasaurus elesitaiensis",
    "Erlikosaurus andrewsi", "Anzu wyliei", "Citipati osmolskae", "Khaan mckennai",
    "Conchoraptor gracilis", "Caudipteryx zoui",

    # Ceratopsia
    "Triceratops horridus", "Triceratops prorsus", "Torosaurus latus",
    "Chasmosaurus belli", "Chasmosaurus russelli", "Anchiceratops ornatus",
    "Arrhinoceratops brachyops", "Kosmoceratops richardsoni", "Utahceratops gettyi",
    "Pentaceratops sternbergii", "Centrosaurus apertus", "Styracosaurus albertensis",
    "Pachyrhinosaurus lakustai", "Pachyrhinosaurus canadensis", "Nasutoceratops titusi",
    "Einiosaurus procurvicornis", "Diabloceratops eatoni", "Wendiceratops pinhornensis",
    "Medusaceratops lokii", "Sinoceratops zhuchengensis", "Protoceratops andrewsi",
    "Psittacosaurus mongoliensis"
]

mIMperSP = 200
maxTotalSpecies = len(nonNegotiableDinos) + 100

def getHybridDinos(limit):
    print("Building hybrid species list...")
    dinoList = list(nonNegotiableDinos)

    if len(dinoList) >= limit:
        return dinoList[:limit]

    wikiURL = "https://en.wikipedia.org/w/api.php"
    headers = {
        "User-Agent": f"dinodatafinder/1.0 (https://kaggle.com/{kUser}) Python-requests"
    }
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": "Category:Dinosaur_genera",
        "cmlimit": 500,
        "cmtype": "page",
        "format": "json"
    }

    resp = requests.get(wikiURL, params=params, headers=headers)
    resp.raise_for_status()
    rawList = [page['title'] for page in resp.json()['query']['categorymembers']]

    excludeWords = [
        "acta", "journal", "annals", "bulletin", "list", "formation",
        "museum", "paleont", "suidae", "family", "cf.", "indet",
        "<span>", "http", "doi"
    ]

    for name in rawList:
        nameClean = name.strip()
        parts = nameClean.split()

        if not (1 <= len(parts) <= 2):
            continue
        if not parts[0][0].isupper():
            continue

        lowerName = nameClean.lower()
        if any(word in lowerName for word in excludeWords):
            continue
        if lowerName.endswith("idae") or lowerName.endswith("inae") or lowerName.endswith("osauria"):
            continue

        if nameClean not in dinoList:
            dinoList.append(nameClean)
            if len(dinoList) >= limit:
                break

    print(f"Total target species: {len(dinoList)}")
    return dinoList

speciesList = getHybridDinos(maxTotalSpecies)

print("loading CLIP")
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print("CLIP loaded successfully.")

junkTags = [
    'toy', 'plastic', 'figure', 'game', 'render', '3d', 'plush',
    'sculpture', 'lego', 'meme', 'origami', 'suit', 'costume',
    'animatronic', 'illustration', 'drawing', 'painting', 'cartoon',
    'diorama', 'taxidermy', 'human', 'tourist', 'graffiti', 'logo'
]

def downloadSingleImage(imgUrl, imgPath, session):
    try:
        time.sleep(0.1)
        imgResp = session.get(imgUrl, timeout=(5, 10))
        if imgResp.status_code == 200:
            if not os.path.exists(imgPath):
                with open(imgPath, "wb") as f:
                    f.write(imgResp.content)
                return True
    except Exception:
        pass
    return False

def scrapeDino(species, maxRes):
    folderName = species.replace(" ", "_")
    fPath = os.path.join(dataDir, folderName)
    os.makedirs(fPath, exist_ok=True)

    existFiles = [f for f in os.listdir(fPath) if f.endswith('.jpg')]
    if len(existFiles) >= maxRes:
        print(f"\n[{species}] Checkpoint active: {len(existFiles)} images found. Skipping download.")
        return

    print(f"\n[{species}] scraping in progress...")

    url = "https://commons.wikimedia.org/w/api.php"
    headers = {
        "User-Agent": f"dinodatafinder/1.0 (https://kaggle.com/{kUser}) Python-requests",
        "Referer": "https://commons.wikimedia.org/"
    }

    session = requests.Session()
    session.headers.update(headers)

    saved = len(existFiles)
    skipped = 0
    searchQueries = [f"{species} skeleton", f"{species} fossil", f"{species} specimen", f"{species}"]
    downloadTasks = []

    try:
        for query in searchQueries:
            if saved + len(downloadTasks) >= maxRes:
                break

            srOffset = 0
            while (saved + len(downloadTasks)) < maxRes and srOffset < 300:
                params = {
                    "action": "query",
                    "generator": "search",
                    "gsrsearch": query,
                    "gsrnamespace": "6",
                    "gsrlimit": 50,
                    "gsroffset": srOffset,
                    "prop": "imageinfo",
                    "iiprop": "url|extmetadata",
                    "format": "json"
                }

                resp = session.get(url, params=params, timeout=10).json()

                if "query" in resp and "pages" in resp["query"]:
                    pages = resp["query"]["pages"]
                    for pId, pInfo in pages.items():
                        if (saved + len(downloadTasks)) >= maxRes:
                            break
                        if "imageinfo" in pInfo:
                            info = pInfo['imageinfo'][0]
                            imgURL = info['url']
                            meta = info.get('extmetadata', {})

                            pathParse = urlparse(imgURL).path.lower()
                            if not any(pathParse.endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.webp']):
                                continue

                            metaText = ""
                            if 'ImageDescription' in meta:
                                metaText += str(meta['ImageDescription'].get('value', '')).lower()
                            if 'Categories' in meta:
                                metaText += str(meta['Categories'].get('value', '')).lower()

                            if any(j in metaText for j in junkTags):
                                skipped += 1
                                continue

                            targetPath = os.path.join(fPath, f"{saved + len(downloadTasks):03d}.jpg")
                            downloadTasks.append((imgURL, targetPath))
                    srOffset += 50
                else:
                    break

        if downloadTasks:
            with ThreadPoolExecutor(max_workers=10) as executor:
                futures = [executor.submit(downloadSingleImage, imgUrl, targetPath, session) for imgUrl, targetPath in downloadTasks]
                for future in futures:
                    if future.result():
                        saved += 1

        print(f"[{species}] Total valid images: {saved}. Skipped {skipped} via metadata.")
    except Exception as e:
        print(f"Error scraping {species}: {e}")

def filterIM(species):
    folderName = species.replace(" ", "_")
    fPath = os.path.join(dataDir, folderName)
    if not os.path.exists(fPath):
        return

    print(f"[{species}] Filtering images...")
    broken = 0
    badVis = 0

    prompts = [
        f"A clear, isolated photograph of a single complete museum dinosaur skeleton of a {species}",
        "A scientific diagram, taxonomic tree, family tree chart, or text-heavy graphic",
        "An illustration, painting, digital render, or children's book page of a dinosaur",
        "A collage, museum wall layout, or photograph containing multiple different dinosaur skeletons",
        "A close-up macro shot of a single isolated bone, individual tooth, or detached claw"
    ]

    tokens = clip.tokenize(prompts).to(device)
    allFiles = [f for f in os.listdir(fPath) if f.endswith('.jpg')]

    batchSize = 32
    for i in range(0, len(allFiles), batchSize):
        batchFiles = allFiles[i:i + batchSize]
        batchImgs = []
        batchPaths = []

        for fName in batchFiles:
            imgPath = os.path.join(fPath, fName)
            try:
                img = Image.open(imgPath).convert("RGB")
                batchImgs.append(preprocess(img))
                batchPaths.append(imgPath)
            except Exception:
                os.remove(imgPath)
                broken += 1

        if not batchImgs:
            continue

        imageTensor = torch.stack(batchImgs).to(device)

        with torch.no_grad():
            logits, _ = model(imageTensor, tokens)
            probs = logits.softmax(dim=-1).cpu().numpy()

        for idx, p in enumerate(probs):
            if p.argmax() != 0:
                os.remove(batchPaths[idx])
                badVis += 1

    print(f"[{species}] done. Removed {broken} broken, {badVis} visually irrelevant items.")

def kagglePush():
    print("\nprepping upload via kagglehub...")
    try:
        handle = f"{kUser}/{kSlug}"
        print(f"Pushing to Kaggle dataset handle: {handle}")

        kagglehub.dataset_upload(
            handle=handle,
            local_dataset_dir=dataDir,
            version_notes="new set"
        )
        print("Upload complete! Check your Kaggle profile.")
    except Exception as e:
        print(f"Kagglehub upload failed: {e}")

if __name__ == "__main__":
    for dino in speciesList:
        scrapeDino(dino, mIMperSP)
        filterIM(dino)
    kagglePush()

In [ ]:
# CONFIGURATIONS
imagesizing = (224, 224, 3)    # PyTorch image sizing requirement
confidencelevel = (0.85, 0.50) # Confidence thresholds: if confidencelevel, the level of confidence the model has -
                               # in its prediction is 0.85/85% or more, it confirms that exact species. -
                               # Otherwise, graceful fallback is triggered (e.g. Ouranosaurus would simply be -
                               # classified as "unidentified basal hadrosauriform" [or simpler if needbe].)

# safeguard-1: confirms tuples containing image sizing and confidence intervals.
def safeguard1():
  print(imagesizing , confidencelevel)

safeguard1()

# requesting data from the Paleobiology Database's (PBDB) live API; I'm using -
# only accepted species of dinosaur for this project.

import requests
DINO_INFO_url = "https://paleobiodb.org/data1.2/taxa/list.json?base_name=Dinosauria&status=accepted"
def getdinosauria(DINO_INFO_url):
  response = requests.get(DINO_INFO_url)
  dinodata = response.json()
  return dinodata

dinodata = getdinosauria(DINO_INFO_url)
print(dinodata["records"][65])

# parses through data

dinotax2 = {}
for records in dinodata["records"]:
  txnID = records["oid"]
  dinotax2[txnID] = {
      "name": records.get("nam"),
      "rank": records.get("rnk"),
      "parent": records.get("par")
  }

def fallingback(txnID):
  fallbackranks1 = ["family" , "genus" , "superfamily" , "subfamily" , "infraorder"]

  records2 = dinotax2.get(txnID)
  if not records2:
    return "UNKNOWN TAXON" , "UNKNOWN RANK"

  parentID = records2.get("parent")

  while parentID:
    parentrnk = dinotax2.get(parentID)
    if not parentrnk:
      break

    currentrank = parentrnk.get("rank")
    currentname = parentrnk.get("name")

    if currentrank in fallbackranks1:
        return currentname, currentrank

    parentID = parentrnk.get("parent")

  return "Unidentified Dinosauria", "clade"


# safeguard
print(f"{len(dinotax2)} taxons identified")

samplekey1 = list(dinotax2.keys())[67]
print("random taxon:" , dinotax2[samplekey1])

# conditionals concerning previously defined confidence thresholds, classifying -
# specimens, and fallback
highconf = confidencelevel[0]
lowconf = confidencelevel[1]


# how the model guesses
def guessing (species, confscore, txnID):
  if confscore >= highconf:
    print(f"{confscore * 100:.1f}% confident of {species}")
    return species, confscore, None
  elif confscore >= lowconf:
    fallbackname , fallbackrank = fallingback(txnID)
    print(f"{confscore * 100:.1f}% confident of {fallbackname} in {fallbackrank}")
    return fallbackname, confscore, fallbackrank
  else:
    fallbackname , fallbackrank = fallingback(txnID)
    print(f"{confscore * 100:.1f}% confidence, unable to identify; unidentified {fallbackrank} ({fallbackname})")
    return None, confscore, fallbackrank

# another safeguard
sample_key = list(dinotax2.keys())[235]
sample_species_name = dinotax2[sample_key]["name"]
guessing(sample_species_name, 0.28, sample_key)
print("THE ABOVE IS A SAMPLE GUESS!!")

# neural network, image processing

import torch
import torch.nn as neuralnetwork
import torchvision.models as TVmodels
import torchvision.datasets as dataset
from torchvision import transforms
from torch.utils.data import DataLoader

#pytorch standard config, training optimization to avoid model from memorizing training images (aka overfitting)

datatransformation = transforms.Compose([
    transforms.Resize((256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

datapath = path
training = dataset.ImageFolder(root = datapath, transform = datatransformation)
speciesamt = len(training.classes)

# another safeguard
print(f"{speciesamt} species found for training")

# data loader
loader = DataLoader(training, batch_size = 50, shuffle  = True)

# resnet model!!

RESNETMODEL = TVmodels.resnet18(weights = TVmodels.ResNet18_Weights.DEFAULT)

# dynamic count
features = RESNETMODEL.fc.in_features
RESNETMODEL.fc = neuralnetwork.Linear(features , speciesamt)
# CUDA
CUDA = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESNETMODEL = RESNETMODEL.to(CUDA)
# loss function and optimization
crit = neuralnetwork.CrossEntropyLoss()
optimizer = torch.optim.AdamW(RESNETMODEL.parameters(), lr = 0.0001, weight_decay = 0.01)
# training loop

epochsnum = 25
# another safeguard
print(f"training on {CUDA}")


for epoch in range(epochsnum):
  RESNETMODEL.train()
  runningloss = 0.0
  correctpredictions = 0
  totalpredictions = 0
  for inputs, labels in loader:
    inputs = inputs.to(CUDA)
    labels = labels.to(CUDA)
    optimizer.zero_grad()
    outputs = RESNETMODEL(inputs)
    loss = crit(outputs, labels)
    loss.backward()
    optimizer.step()

    # batch, summary metrics
    runningloss += loss.item() * labels.size(0)
    _, predicts = torch.max(outputs, dim = 1)
    correctpredictions += torch.sum(predicts == labels).item()
    totalpredictions += labels.size(0)

  epochloss = runningloss / totalpredictions
  epochaccuracy = (correctpredictions / totalpredictions) * 100


# another safeguard, saving
  print(f"Epoch [{epoch+1}/{epochsnum}] - Loss: {epochloss:.4f} - Accuracy: {epochaccuracy:.2f}%")

print("trained")

torch.save(RESNETMODEL.state_dict(), "resnet18DINOSAUR.pth")
print("saved to resnet18DINOSAUR.pth")

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"shayanr10","key":"9d3976c66295ce2c10598814439fdf60"}'}

In [ ]:
import os
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists('kaggle.json'):
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 600)
    print("New kaggle.json successfully installed!")
else:
    print("Please upload your kaggle.json file using the widget above.")

New kaggle.json successfully installed!
